In [2]:
import numpy as np

image = np.array([
    [1, 1, 1, 0],
    [0, 1, 1, 1],
    [0, 0, 1, 1],
    [1, 1, 0, 0]
])

kernel = np.array([
    [1, 0],
    [0, 1]
])

patch = image[0:2, 0:2]
result = np.sum(patch*kernel)
print(result)

2


In [4]:
for i in range(3):
    for j in range(3):
        print((i,j))

(0, 0)
(0, 1)
(0, 2)
(1, 0)
(1, 1)
(1, 2)
(2, 0)
(2, 1)
(2, 2)


In [ ]:
def convolve(image, kernel):
    output_size = image.shape[0] - kernel.shape[0] + 1
    output = np.zeros((output_size, output_size))

    for i in range(output_size):
        for j in range(output_size):
            output[i, j] = np.sum(image[i:i+kernel.shape[0], j:j+kernel.shape[1]] * kernel)
    
    return output

def ReLU(x):
    output = np.maximum(0, x)
    return output

print(ReLU(convolve(image, kernel)))



[[2. 2. 2.]
 [0. 2. 2.]
 [1. 0. 1.]]
[[2. 2. 2.]
 [0. 2. 2.]
 [1. 0. 1.]]


In [35]:
# POOLING
input = np.array([[1,5,2,0],
                  [7,3,4,1],
                  [6,8,0,2],
                  [1,9,5,3]])
def max_pool(feature_map):
    output = np.zeros((feature_map.shape[0]//2,feature_map.shape[1]//2))
    x = 0
    for i in range(0, feature_map.shape[0], 2):
        y = 0
        for j in range(0, feature_map.shape[1], 2):
            patch = feature_map[i:i+2, j:j+2]
            output[x, y] = np.max(patch)
            y += 1
        x += 1
    return output

def flatten(x):
    return np.reshape(x, -1)

print(flatten(max_pool(input)))

[7. 4. 9. 5.]


In [36]:
x = np.array([7,4,9,5])

weights = np.array([
    [0.2,0.5,0.1],
    [0.3,0.7,0.2],
    [0.8,0.1,0.9],
    [0.4,0.6,0.3]
])

bias = np.array([0.1,0.2,0.3])

output = np.dot(x, weights) + bias

print(output)

[11.9 10.4 11.4]


In [38]:
def softmax(x):
    x = x - np.max(x)
    exp = np.exp(x)
    normalised = exp / np.sum(exp)
    return normalised

scores = np.array([2,1,0])

print(softmax(scores))


[0.66524096 0.24472847 0.09003057]


In [40]:
def cross_entropy(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-7, 1-1e-7)
    return -np.sum(y_true * np.log(y_pred))

y_true = np.array([0,1,0])

y_pred = np.array([0.7,0.2,0.1])

print(cross_entropy(y_true, y_pred))

1.6094379124341003


In [46]:
x = 2
w = 3
y = 10

learning_rate = 0.1

gradient = -2 * x * (y - x*w)

w = w - learning_rate * gradient

print(w)

4.6


In [47]:
a = np.array([1,2,3])
b = np.array([4,5])

np.outer(a, b)

array([[ 4,  5],
       [ 8, 10],
       [12, 15]])

In [49]:
def dense_backward(x, W, d_scores):
    dW = np.outer(x, d_scores)
    db = d_scores
    dx = np.dot(d_scores, W.T)

    return dW, db, dx

x = np.array([7,4,9,5])

W = np.random.randn(4,3)

d_scores = np.array([0.7,-0.8,0.1])
dW, db, dx = dense_backward(x, W, d_scores)
print(dW.shape)
print(db.shape)
print(dx.shape)

(4, 3)
(3,)
(4,)


In [59]:
def dense_forward(x, W, b):
    return np.dot(x, W) + b


np.random.seed(0)
W = np.random.randn(4,3) * 0.01
b = np.zeros(3)
x = np.array([7,4,9,5])
y_true = np.array([0,1,0])

scores = dense_forward(x, W, b)
y_pred = softmax(scores)
loss = cross_entropy(y_true, y_pred)

print(y_pred)
print(loss)

d_scores = y_pred - y_true
dW, db, dx = dense_backward(x, W, d_scores)

learning_rate = 0.01

W -= learning_rate * dW
b -= learning_rate * db

scores = dense_forward(x, W, b)
y_pred = softmax(scores)
loss = cross_entropy(y_true, y_pred)

print(y_pred)
print(loss)

d_scores = y_pred - y_true
dW, db, dx = dense_backward(x, W, d_scores)

learning_rate = 0.01

W -= learning_rate * dW
b -= learning_rate * db

scores = dense_forward(x, W, b)
y_pred = softmax(scores)
loss = cross_entropy(y_true, y_pred)

print(y_pred)
print(loss)



[0.38495705 0.30805177 0.30699118]
1.1774874287103907
[0.14259654 0.72736792 0.13003554]
0.31832285445286634
[0.08096791 0.84358388 0.07544821]
0.17009593803978082


In [60]:
def relu_backward(x, d_out):
    d_relu = d_out * (x>0)
    return d_relu

x = np.array([-3,4,-1,7])
d_out = np.array([1,1,1,1])

relu_backward(x, d_out)

array([0, 1, 0, 1])

In [64]:
def max_pool_backward(feature_map,d_out):
    dx = np.zeros_like(feature_map)
    x = 0

    for i in range(0, feature_map.shape[0], 2):
        y = 0
        for j in range(0, feature_map.shape[1], 2):
            patch = feature_map[i:i+2, j:j+2]
            mask = (patch == np.max(patch))
            dx[i:i+2, j:j+2] = mask * d_out[x, y]
            y+=1
        x+=1

    return dx

feature_map = np.array([
    [1,5,2,0],
    [7,3,4,1],
    [6,8,0,2],
    [1,9,5,3]
])

d_out = np.array([
    [10,20],
    [30,40]
])

max_pool_backward(feature_map, d_out)



array([[ 0,  0,  0,  0],
       [10,  0, 20,  0],
       [ 0,  0,  0,  0],
       [ 0, 30, 40,  0]])

In [ ]:
def conv_backward(image, kernel, d_out):
    dKernel = np.zeros_like(kernel)
    dInput = np.zeros_like(image)

    for i in range(d_out.shape[0]):
        for j in range(d_out.shape[0]):
            patch = image[i:i+kernel.shape[0], j:j+kernel.shape[1]]
            dKernel += patch * d_out[i, j]
            dInput[i:i+kernel.shape[0], j:j+kernel.shape[1]] += kernel * d_out[i, j]